# Neural networks
There are many machine learning techniques that works fine for tabular data, at the moment, we have seen popular approaches like decision trees, random forests, support vector machines or naive Bayes classifiers. There is a special algorithm that has taken center stage some years ago, yes I'm talking about neural networks probably the most popular ML technique, specially famous for some implementations like convolutional networks, recurrent neural networks and of course the architecture responsible for creating large language models like GPT, the transformers architecture. 

## Components
- **Neural node:** A representation of a human neuron, that performs a linear transformation to the input and applies an activation function to define the output.
- **Activation function:** A function that takes the linear combination of the input features and adds non-linearity.
- **Layers:** Is a set of neural nodes that takes as input the outputs of the previous layer or the model inputs when is the first layer.


## How does inference is performed?
1. Take $\vec{x}$ input vector to the first neuron and apply the linear transforming. Where $W_{1,1}$ is the weights matrix and $b_{1,1}$ the bias coefficient for neuron 1 at layer 1.

<center>
    $h_{1,1} = W_{1,1}\vec{x} + b_{1,1}$
</center>

2. Evaluate the activation function $a(x)$ using the coefficient $h_{1,1}$ to calculate the neuron output $y_{1,1}$.

<center>
    $y_{1,1} = a(h_{1,1})$
</center>

3. Repeat the process for every neuron at layer 1 to obtain the output vector $\vec{x}_{1}$.

<center>
    $\vec{x}_{1}^{T} = (y_{1,1}, y_{1,2}, ... , y_{1,n})$
</center>

4. Once $\vec{x}_{1}$ was calculated, repeat steps 1 to 3 for every neuron at second layer using $\vec{x}_{1}$ as input.
5. Repeat steps 1 to 4 for every layer until achiving output layer, every neuron output is represented by the following relantionship:

<center>
    $y_{i,j} = a(W_{i-1, j} {{\vec{x}}_{i-1}}_{j} + b_{i,j})$
</center>

6. When achieving the output layer, activation function $a(x)$ is replaced by a special function dependant of the task (regression, binary classification or multiclass classification) is used to get the outcome $\bar{y}$.

This algorithm is called **feed forward step**.


In [1]:
from pandas import read_csv
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = ExtraTreesClassifier(n_estimators=20, criterion="entropy", max_depth=10, random_state=123)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

Accuracy: 81.750000 %
Weighted F1-Score: 81.373105 %


In [2]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import ExtraTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = ExtraTreeRegressor(criterion="squared_error", random_state=123)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")

Mean squared error: 1712235348.274637
Mean absolute error: 6907.316468
